# Web-PDE-LLM: Multi-Agent Cardiac PDE Workflow

This notebook demonstrates the full **5-agent pipeline** for generating GPU-accelerated
WebGL cardiac simulations from natural language PDE descriptions.

| Step | Agent | Role |
|------|-------|------|
| 0 | **Clarifier** | Validate / complete the user spec |
| 1 | **Parse** | Spec → structured JSON |
| 2 | **Coding** | JSON + skeleton → GLSL shader |
| 3 | **Debug** | Fix compilation / logical errors |
| 4 | **Validation** | Static physical correctness check |

Supported cardiac models:
- **Fenton-Karma 3V** (u, v, w) — default
- **Aliev-Panfilov 2V** (u, v)

For parallel execution of both models run:
```bash
python pipeline.py --all
```

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import glob
import logging
from pathlib import Path
from ollama import chat

import clarifier_agent
import parse_agent
import coding_agent
import debug_agent
import validation_agent
import pde_descriptions
import system_prompt as sp
from config import Config

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
cfg = Config()
print('Config:', cfg)

In [ ]:
# ── Shared helpers ──────────────────────────────────────────────────────────

def strip_fences(text, lang=''):
    text = text.strip()
    for fence in (f'```{lang}', '```'):
        if text.startswith(fence):
            text = text[len(fence):]
            break
    if text.endswith('```'):
        text = text[:-3]
    return text.strip()


def llm(model, prompt, system=''):
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': prompt})
    return chat(model=model, messages=msgs).message.content


# Read WebGL error log if present (written by browser test harness)
log_content = ''
for lf in glob.glob('*.log'):
    log_content += Path(lf).read_text(encoding='utf-8')

print(f'Log content: {len(log_content)} chars')

---
## Model 1: Fenton-Karma 3V
### Step 0 — Clarifier

In [ ]:
tau_d = 0.5714  # change this to explore different FK variants
fk_input = pde_descriptions.fk_description.format(tau_d=tau_d)

fk_clarify_raw = strip_fences(
    llm(cfg.clarifier_model, clarifier_agent.clarifier_prompt.format(user_input=fk_input)),
    'json'
)
fk_clarified = json.loads(fk_clarify_raw)
print('Status:', fk_clarified['status'])
if fk_clarified.get('missing'):
    print('Missing fields:', fk_clarified['missing'])
    print('Questions for user:', fk_clarified.get('questions'))

### Step 1 — Parse

In [ ]:
fk_spec = fk_clarified.get('clarified_spec', fk_input)

fk_parse_raw = strip_fences(
    llm(cfg.parse_model, parse_agent.parse_prompt.format(user_input=fk_spec)),
    'json'
)
fk_parsed = json.loads(fk_parse_raw)
Path('outputs/fenton_karma').mkdir(parents=True, exist_ok=True)
Path('outputs/fenton_karma/parsed_resp.json').write_text(json.dumps(fk_parsed, indent=2))
print(json.dumps(fk_parsed, indent=2))

### Step 2 — Coding

In [ ]:
fk_skeleton = Path('coding_skeleton.frag').read_text()

fk_shader = strip_fences(
    llm(
        cfg.code_model,
        coding_agent.coding_prompt.format(
            PDEs=fk_parsed['PDEs'],
            parameter_values=json.dumps(fk_parsed.get('parameter_values', {}), indent=2),
            coding_skeleton=fk_skeleton,
        ),
        system=sp.system_prompt,
    ),
    'glsl'
)
print(fk_shader[:600], '\n...')

### Step 3 — Debug

In [ ]:
if log_content.strip():
    fk_shader = strip_fences(
        llm(
            cfg.debug_model,
            debug_agent.debug_prompt.format(shader_codes=fk_shader, log_info=log_content),
        ),
        'glsl'
    )
    print('Debug pass complete.')
else:
    print('No log file — skipping debug pass.')

### Step 4 — Validation

In [ ]:
fk_val_raw = strip_fences(
    llm(
        cfg.validation_model,
        validation_agent.validation_prompt.format(
            shader_code=fk_shader,
            parameter_values=json.dumps(fk_parsed.get('parameter_values', {}), indent=2),
            num_state_vars=fk_parsed.get('number_of_state_variables', 3),
            model_notes=fk_parsed.get('notes') or '',
        ),
    ),
    'json'
)
fk_validation = json.loads(fk_val_raw)
print(json.dumps(fk_validation, indent=2))

# Auto-fix if validation fails
if fk_validation.get('status') == 'fail':
    issues = '\n'.join(fk_validation.get('issues', []))
    suggestion = fk_validation.get('suggestion', '')
    fk_shader = strip_fences(
        llm(
            cfg.debug_model,
            debug_agent.debug_prompt.format(
                shader_codes=fk_shader,
                log_info=f'Validation failed:\n{issues}\nSuggestion: {suggestion}',
            ),
        ),
        'glsl'
    )
    print('Auto-fix applied.')

In [ ]:
fk_out = Path('outputs/fenton_karma/march_shader.frag')
fk_out.write_text(fk_shader, encoding='utf-8')
print(f'Saved: {fk_out}')

---
## Model 2: Aliev-Panfilov 2V
### Step 0 — Clarifier

In [ ]:
ap_input = pde_descriptions.ap_description

ap_clarify_raw = strip_fences(
    llm(cfg.clarifier_model, clarifier_agent.clarifier_prompt.format(user_input=ap_input)),
    'json'
)
ap_clarified = json.loads(ap_clarify_raw)
print('Status:', ap_clarified['status'])
if ap_clarified.get('missing'):
    print('Missing:', ap_clarified['missing'])

### Step 1 — Parse

In [ ]:
ap_spec = ap_clarified.get('clarified_spec', ap_input)

ap_parse_raw = strip_fences(
    llm(cfg.parse_model, parse_agent.parse_prompt.format(user_input=ap_spec)),
    'json'
)
ap_parsed = json.loads(ap_parse_raw)
Path('outputs/aliev_panfilov').mkdir(parents=True, exist_ok=True)
Path('outputs/aliev_panfilov/parsed_resp.json').write_text(json.dumps(ap_parsed, indent=2))
print(json.dumps(ap_parsed, indent=2))

### Step 2 — Coding

In [ ]:
ap_skeleton = Path('ap_coding_skeleton.frag').read_text()

ap_shader = strip_fences(
    llm(
        cfg.code_model,
        coding_agent.coding_prompt.format(
            PDEs=ap_parsed['PDEs'],
            parameter_values=json.dumps(ap_parsed.get('parameter_values', {}), indent=2),
            coding_skeleton=ap_skeleton,
        ),
        system=sp.system_prompt,
    ),
    'glsl'
)
print(ap_shader[:600], '\n...')

### Step 3 — Debug

In [ ]:
if log_content.strip():
    ap_shader = strip_fences(
        llm(
            cfg.debug_model,
            debug_agent.debug_prompt.format(shader_codes=ap_shader, log_info=log_content),
        ),
        'glsl'
    )
    print('Debug pass complete.')
else:
    print('No log file — skipping debug pass.')

### Step 4 — Validation

In [ ]:
ap_val_raw = strip_fences(
    llm(
        cfg.validation_model,
        validation_agent.validation_prompt.format(
            shader_code=ap_shader,
            parameter_values=json.dumps(ap_parsed.get('parameter_values', {}), indent=2),
            num_state_vars=ap_parsed.get('number_of_state_variables', 2),
            model_notes=ap_parsed.get('notes') or '',
        ),
    ),
    'json'
)
ap_validation = json.loads(ap_val_raw)
print(json.dumps(ap_validation, indent=2))

if ap_validation.get('status') == 'fail':
    issues = '\n'.join(ap_validation.get('issues', []))
    suggestion = ap_validation.get('suggestion', '')
    ap_shader = strip_fences(
        llm(
            cfg.debug_model,
            debug_agent.debug_prompt.format(
                shader_codes=ap_shader,
                log_info=f'Validation failed:\n{issues}\nSuggestion: {suggestion}',
            ),
        ),
        'glsl'
    )
    print('Auto-fix applied.')

In [ ]:
ap_out = Path('outputs/aliev_panfilov/march_shader.frag')
ap_out.write_text(ap_shader, encoding='utf-8')
print(f'Saved: {ap_out}')

---
## Summary

In [ ]:
print('=' * 58)
print('  RESULTS')
print('=' * 58)
for name, val, path in [
    ('Fenton-Karma 3V',   fk_validation, fk_out),
    ('Aliev-Panfilov 2V', ap_validation, ap_out),
]:
    status = val.get('status', '?')
    checks = val.get('checks', {})
    n_pass = sum(1 for v in checks.values() if v)
    n_total = len(checks)
    print(f'  {name:<22}  {status:<6}  checks {n_pass}/{n_total}  →  {path}')
print('=' * 58)
print()
print('Tip: run both models in parallel:')
print('  python pipeline.py --all')